In [1]:
import os
import cv2
import numpy as np
import pandas as pd
from tqdm import tqdm
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

In [ ]:
# =======================
# STEP 1: PoseLandmarker Setup
# =======================   
base_options = python.BaseOptions(model_asset_path='pose_landmarker.task')
options = vision.PoseLandmarkerOptions(
    base_options=base_options,
    output_segmentation_masks=False,  
    running_mode=vision.RunningMode.IMAGE
)
detector = vision.PoseLandmarker.create_from_options(options)

In [4]:
# =======================
# STEP 2: Function to extract landmarks
# =======================
def extract_landmarks(img_path):
    image = mp.Image.create_from_file(img_path)
    detection_result = detector.detect(image)

    if not detection_result.pose_landmarks:
        return None

    # Get first detected person
    landmarks = detection_result.pose_landmarks[0]
    coords = []
    for lm in landmarks:
        coords.extend([lm.x, lm.y, lm.z, lm.visibility])
    return coords

In [6]:
DATA_DIR = "D:/Dance_Doodle_Dataset"  
data = []
labels = []

for label_name in os.listdir(DATA_DIR):
    folder_path = os.path.join(DATA_DIR, label_name)
    if not os.path.isdir(folder_path):
        continue

    for img_file in tqdm(os.listdir(folder_path), desc=f"Processing {label_name}"):
        img_path = os.path.join(folder_path, img_file)
        try:
            coords = extract_landmarks(img_path)
            if coords:
                data.append(coords)
                labels.append(label_name)
        except:
            pass

data = np.array(data)
labels = np.array(labels)

print("Data shape:", data.shape)
print("Labels shape:", labels.shape)

Processing stretch_right: 100%|██████████| 176/176 [01:01<00:00,  2.87it/s]

Data shape: (1820, 132)
Labels shape: (1820,)


In [7]:
import random
from collections import defaultdict

def balance_dataset(data, labels, seed=42):
    """
    Balances dataset by randomly sampling an equal number of samples for each label.
    """
    random.seed(seed)
    np.random.seed(seed)

    # Group indices by label
    label_to_indices = defaultdict(list)
    for idx, lbl in enumerate(labels):
        label_to_indices[lbl].append(idx)

    # Find the minimum number of samples in any class
    min_count = min(len(idxs) for idxs in label_to_indices.values())
    print(f"Minimum samples per class: {min_count}")

    # Collect balanced indices
    balanced_indices = []
    for lbl, idxs in label_to_indices.items():
        chosen = random.sample(idxs, min_count)
        balanced_indices.extend(chosen)

    random.shuffle(balanced_indices)

    # Create balanced arrays
    balanced_data = data[balanced_indices]
    balanced_labels = labels[balanced_indices]

    return balanced_data, balanced_labels


# === Apply balancing after your data collection ===
balanced_data, balanced_labels = balance_dataset(data, labels)

print("Balanced data shape:", balanced_data.shape)
print("Balanced labels shape:", balanced_labels.shape)


Minimum samples per class: 145
Balanced data shape: (1450, 132)
Balanced labels shape: (1450,)


In [8]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# Encode labels
encoder = LabelEncoder()
y_encoded = encoder.fit_transform(balanced_labels)

X_train, X_test, y_train, y_test = train_test_split(
    balanced_data, y_encoded, test_size=0.2, random_state=42
)

In [10]:
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report

# Define classifiers
models = {
    "Random Forest": RandomForestClassifier(n_estimators=150, random_state=42),
    "Logistic Regression": LogisticRegression(max_iter=2000),
    "SVM (RBF Kernel)": SVC(kernel='rbf'),
    "KNN": KNeighborsClassifier(n_neighbors=3),
    "Decision Tree": DecisionTreeClassifier(),
    "Gradient Boosting": GradientBoostingClassifier()
}

results = {}
print("🔍 Model Training & Evaluation Details:\n")

for name, model in models.items():
    print("=" * 60)
    print(f"▶ Training {name}...")
    model.fit(X_train, y_train)
    print(f"✅ Finished training {name}.\n")

    # Predictions & Metrics
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    results[name] = acc

    print(f"📊 {name} Accuracy: {acc:.4f}")
    print(f"📄 Classification Report for {name}:\n")
    print(classification_report(y_test, y_pred, target_names=encoder.classes_))
    print("=" * 60 + "\n")

# Show best model
best_model_name = max(results, key=results.get)
print("🏆 Best Model:", best_model_name, f"({results[best_model_name]:.4f} accuracy)")


🔍 Model Training & Evaluation Details:

▶ Training Random Forest...
✅ Finished training Random Forest.

📊 Random Forest Accuracy: 1.0000
📄 Classification Report for Random Forest:

                   precision    recall  f1-score   support

        cool_arms       1.00      1.00      1.00        25
      crossy_play       1.00      1.00      1.00        30
 happy_stand_left       1.00      1.00      1.00        32
happy_stand_right       1.00      1.00      1.00        27
       open_wings       1.00      1.00      1.00        24
       ready_pose       1.00      1.00      1.00        36
          shh_fun       1.00      1.00      1.00        34
      silly_boxer       1.00      1.00      1.00        25
     stretch_left       1.00      1.00      1.00        27
    stretch_right       1.00      1.00      1.00        30

         accuracy                           1.00       290
        macro avg       1.00      1.00      1.00       290
     weighted avg       1.00      1.00      1.00  

In [13]:
import joblib
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

# Encode labels
encoder = LabelEncoder()
y_encoded = encoder.fit_transform(balanced_labels)

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    balanced_data, y_encoded, test_size=0.2, random_state=42
)

# Train Decision Tree
dt_model = DecisionTreeClassifier()
dt_model.fit(X_train, y_train)

# Save model and encoder
joblib.dump(dt_model, "c:/NeuroNurture/Javafest_agor/Games/dance_doodle_decision_tree_model.pkl")
joblib.dump(encoder, "c:/NeuroNurture/Javafest_agor/Games/dance_doodle_label_encoder.pkl")

print("Decision Tree model and label encoder saved!")


Decision Tree model and label encoder saved!
